# MPMAvatar — SDS Physics Training
Run cells top to bottom. Each section is self-contained.

## Cell 1 — Check GPU

In [ ]:
import subprocess, os, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

cc = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cc[0]}.{cc[1]}"
os.environ['FORCE_CUDA'] = '1'
print(f"CUDA arch: sm_{cc[0]}{cc[1]}")


## Cell 2 — Clone Repo

In [ ]:
import os

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
GITHUB_USER = "poudelsaroj"   # change if needed

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/mpm_avatar_vds.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git stash && git pull
print("Repo ready:", REPO_DIR)


## Cell 3 — Install Python Dependencies
Run once per session. Uses numpy 2.x throughout (Wan works fine despite the version warning).

In [ ]:
# Step 1: upgrade pandas first (2.2+ supports numpy 2.x; 2.1.x does not)
!pip install -q "pandas>=2.2"

# Step 2: core deps
!pip install -q \
    "numpy>=2.0" \
    scipy scikit-learn \
    Pillow tqdm pyyaml wandb einops plyfile trimesh \
    smplx roma jaxtyping mediapy imageio safetensors accelerate \
    "warp-lang>=1.0.0" \
    transformers sentencepiece huggingface_hub \
    human-body-prior \
    librosa opencv-python-headless peft easydict ftfy \
    diffusers decord tokenizers imageio-ffmpeg

# Step 3: force numpy 2.x last so nothing downgrades it
!pip install -q --force-reinstall "numpy>=2.0" --no-deps

import numpy as np, sys
print(f"numpy {np.__version__} / Python {sys.version.split()[0]}")
print("⚠  Restart kernel now (Kernel → Restart), then re-run from Cell 1.")


## Cell 4 — Install pytorch3d (~5 min, builds from source)

In [ ]:
import os, torch

# Detect GPU arch — only compile for the current card (halves build time)
cc = torch.cuda.get_device_capability()
arch = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = arch
os.environ["FORCE_CUDA"] = "1"
os.environ["MAX_JOBS"]   = "8"   # parallel C++ jobs — ~10 min on L4, ~3 min on H100
print(f"Building pytorch3d for sm_{cc[0]}{cc[1]} with MAX_JOBS=8 ...")

!pip install -q fvcore iopath
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable" --no-build-isolation

import pytorch3d
print(f"✓ pytorch3d {pytorch3d.__version__}")

## Cell 5 — Build CUDA Extensions (diff_gauss, simple_knn)

In [ ]:
import os, subprocess, sys

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
EXT_DIR  = "/teamspace/studios/this_studio/ext"
os.makedirs(EXT_DIR, exist_ok=True)

def build(name, repo_url, extra_patch=None, editable=True):
    dst = f"{EXT_DIR}/{name}"
    if not os.path.exists(dst):
        subprocess.run(["git","clone","--recurse-submodules", repo_url, dst], check=True)
    if extra_patch:
        extra_patch(dst)
    cmd = [sys.executable,"-m","pip","install","--no-build-isolation","-q"]
    if editable:
        cmd += ["-e", dst]
    else:
        cmd.append(dst)
    subprocess.run(cmd, check=True)
    print(f"{name} done")

def patch_diff_gauss(path):
    # auxiliary.h and rasterizer_impl.h both need <cstdint> for uint32_t / uintptr_t
    for fname in ["cuda_rasterizer/auxiliary.h", "cuda_rasterizer/rasterizer_impl.h"]:
        f = os.path.join(path, fname)
        if not os.path.exists(f):
            continue
        src = open(f).read()
        if "#include <cstdint>" not in src:
            open(f, "w").write("#include <cstdint>\n" + src)
            print(f"  patched {fname}")

build("diff-gaussian-rasterization",
      "https://github.com/slothfulxtx/diff-gaussian-rasterization.git",
      patch_diff_gauss, editable=True)

# simple-knn is a pure C extension — non-editable install avoids .so path issues
build("simple-knn",
      "https://gitlab.inria.fr/bkerbl/simple-knn.git",
      editable=False)

# Verify
subprocess.run([sys.executable,"-c","import diff_gauss; print('✓ diff_gauss OK')"], check=True)
subprocess.run([sys.executable,"-c","import simple_knn; print('✓ simple_knn OK')"], check=True)


## Cell 6 — Verify diffusers WanPipeline (Wan 5B)

In [ ]:
# Upgrade diffusers to latest (needs WanPipeline support added in ~0.32+)
!pip install -q -U "git+https://github.com/huggingface/diffusers"

# Verify WanPipeline is importable
try:
    from diffusers import WanPipeline, AutoencoderKLWan
    import diffusers
    print(f"\u2713 diffusers {diffusers.__version__} — WanPipeline OK")
except ImportError as e:
    print(f"\u2717 WanPipeline not found: {e}")
    print("  Try: pip install git+https://github.com/huggingface/diffusers")


## Cell 7 — Download Wan2.2-TI2V-5B-Diffusers (~22 GB)
Run once, skips if already downloaded.  5B model fits on a single A100/H100 GPU.

In [ ]:
import os
from huggingface_hub import snapshot_download, login

HF_TOKEN     = os.environ.get("HF_TOKEN", "")   # set in Lightning Secrets or paste here
WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_5b_model"
os.makedirs(WAN_CKPT_DIR, exist_ok=True)

# Check if already downloaded (diffusers layout has model_index.json)
if not os.path.exists(f"{WAN_CKPT_DIR}/model_index.json"):
    print("Downloading Wan2.2-TI2V-5B-Diffusers (~22 GB) ...")
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
    snapshot_download(
        repo_id="Wan-AI/Wan2.2-TI2V-5B-Diffusers",
        local_dir=WAN_CKPT_DIR,
        ignore_patterns=["*.md", "*.txt"],
        token=HF_TOKEN or None,
    )
    print("Download complete.")
else:
    print("Already downloaded.")

print("\nContents:")
for name in sorted(os.listdir(WAN_CKPT_DIR)):
    print(f"  {name}")


## Cell 8 — Verify Wan 5B Loads
Quick smoke test: instantiate the pipeline and check all components are present.

In [ ]:
import os, torch
from diffusers import WanPipeline, AutoencoderKLWan

WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_5b_model"

if not os.path.exists(f"{WAN_CKPT_DIR}/model_index.json"):
    print("\u2717 Model not found — run Cell 7 first.")
else:
    print("Checking model components (not loading weights yet) ...")
    required = ["transformer", "vae", "text_encoder", "tokenizer", "scheduler", "model_index.json"]
    all_ok = True
    for name in required:
        path = f"{WAN_CKPT_DIR}/{name}"
        ok = os.path.exists(path)
        if not ok:
            all_ok = False
        print(f"  {'\u2713' if ok else '\u2717'} {name}")

    if all_ok:
        print("\n\u2713 All components present — ready for training.")
    else:
        print("\n\u2717 Missing components — re-run Cell 7.")


## Cell 9 — Download & Place Dataset Files

Download the full dataset TAR from Google Drive, then run this cell to place everything.

```bash
# In Lightning terminal:
pip install gdown
gdown "1d1UXaqhiqOUx1NqggKreGrClldb9dxeR" -O /teamspace/studios/this_studio/mpmavatar_data.tar.gz
tar -xzf /teamspace/studios/this_studio/mpmavatar_data.tar.gz -C /teamspace/studios/this_studio/
```

The TAR extracts to `/teamspace/studios/this_studio/data/` which has the full dataset structure.
Run this cell to copy model files to `pretrained_models/`.


In [ ]:
import os, shutil, zipfile, tarfile, glob

STUDIO     = "/teamspace/studios/this_studio"
STAGING    = f"{STUDIO}/extracted_data"
PRETRAINED = f"{STUDIO}/pretrained_models"
DATA_DIR   = f"{STUDIO}/data"

PLY_DST      = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
AOMAP_DST    = f"{TRACKING_DST}/aomap"
SMPLX_DST    = f"{DATA_DIR}/body_models/smplx"
BODY_DST     = f"{DATA_DIR}/body_models"
A1S1_DST     = f"{DATA_DIR}/a1_s1"

for d in [STAGING, PLY_DST, AOMAP_DST, SMPLX_DST, A1S1_DST]:
    os.makedirs(d, exist_ok=True)

# ── Step 1: Extract archives ──────────────────────────────────────────────────
archives = {
    "MPMAvatar_processed_data.zip": f"{STAGING}/processed_data",
    "models_smplx_v1_1.zip":        f"{STAGING}/smplx",
    "vposer_v1_0.zip":              f"{STAGING}/vposer",
    "mpmavatar_data.tar.gz":        f"{STAGING}/mpmavatar_data",
}

for fname, dst in archives.items():
    src = f"{STUDIO}/{fname}"
    if not os.path.exists(src):
        print(f"  ✗ NOT FOUND: {fname}")
        continue
    if os.path.exists(dst) and os.listdir(dst):
        print(f"  · already extracted: {fname}")
        continue
    os.makedirs(dst, exist_ok=True)
    print(f"  extracting {fname} ...")
    if fname.endswith(".tar.gz"):
        with tarfile.open(src) as tf:
            tf.extractall(dst)
    else:
        with zipfile.ZipFile(src) as zf:
            zf.extractall(dst)
    print(f"    → {dst}")

print("\nExtraction done.\n")

# ── Step 2: Find & copy files by name ────────────────────────────────────────

def find_file(root, name):
    """Recursively find first file matching name under root."""
    for path in glob.glob(f"{root}/**/{name}", recursive=True):
        return path
    return None

def find_files(root, pattern):
    """Recursively find all files matching glob pattern under root."""
    return glob.glob(f"{root}/**/{pattern}", recursive=True)

def cp(src, dst, label):
    if not src or not os.path.exists(src):
        print(f"  ✗ NOT FOUND  {label}")
        return False
    if os.path.exists(dst):
        print(f"  · exists     {label}")
        return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  ✓ copied     {label}  ← {os.path.relpath(src, STAGING)}")
    return True

def cp_dir(src_dir, dst_dir, pattern, label):
    """Copy all files matching pattern from src_dir to dst_dir."""
    files = glob.glob(f"{src_dir}/**/{pattern}", recursive=True)
    if not files:
        print(f"  ✗ NOT FOUND  {label}")
        return 0
    n = 0
    for f in files:
        dst = f"{dst_dir}/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy2(f, dst)
            n += 1
    total = len(glob.glob(f"{dst_dir}/{pattern}"))
    print(f"  ✓ {label}: {n} new ({total} total)")
    return n

P = f"{STAGING}/processed_data"
S = f"{STAGING}/smplx"
V = f"{STAGING}/vposer"
D = f"{STAGING}/mpmavatar_data"

# Gaussian model files
cp(find_file(P, "point_cloud.ply"),  f"{PLY_DST}/point_cloud.ply",  "point_cloud.ply")
cp(find_file(P, "verts_offset.npy"), f"{PLY_DST}/verts_offset.npy", "verts_offset.npy")
cp(find_file(P, "cams.npz"),         f"{PLY_DST}/cams.npz",         "cams.npz")
cp(find_file(P, "shadow_net.pt"),    f"{PLY_DST}/shadow_net.pt",    "shadow_net.pt")

# Tracking params
cp_dir(P, TRACKING_DST, "params_*.npz", "params_*.npz")

# AO maps
cp_dir(P, AOMAP_DST, "mesh_cloth_*.png", "aomap/mesh_cloth_*.png")
cp_dir(P, AOMAP_DST, "*.png",            "aomap/*.png")

# Dataset files
cp(find_file(D, "split_idx.npz"),         f"{A1S1_DST}/split_idx.npz",         "split_idx.npz")
cp(find_file(D, "cam_info.json"),         f"{A1S1_DST}/cam_info.json",          "cam_info.json")
cp(find_file(D, "optimized_weights.npy"), f"{A1S1_DST}/optimized_weights.npy",  "optimized_weights.npy")

# SMPLX body model
cp(find_file(S, "SMPLX_NEUTRAL.npz"), f"{SMPLX_DST}/SMPLX_NEUTRAL.npz", "SMPLX_NEUTRAL.npz")

# VPoser (TR00_E096.pt)
cp(find_file(V, "TR00_E096.pt"), f"{BODY_DST}/TR00_E096.pt", "TR00_E096.pt")
# fallback: search everywhere
if not os.path.exists(f"{BODY_DST}/TR00_E096.pt"):
    cp(find_file(STAGING, "TR00_E096.pt"), f"{BODY_DST}/TR00_E096.pt", "TR00_E096.pt (fallback)")

print()


## Cell 10 — Generate Missing Data Files
Generates `cam_info.json`, `a1s1_uv.obj`, `smplx_fitted/` and `split_idx.npz` if missing.

In [ ]:
import os, shutil, glob

STUDIO   = "/teamspace/studios/this_studio"
STAGING  = f"{STUDIO}/extracted_data"
DATA_DIR = f"{STUDIO}/data"
REPO_DIR = f"{STUDIO}/mpm_avatar_vds"

A1S1_DST = f"{DATA_DIR}/a1_s1"

# smplx_fitted/ — copy whole directory
src = f"{STAGING}/mpmavatar_data/data/a1_s1/smplx_fitted"
dst = f"{A1S1_DST}/smplx_fitted"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copytree(src, dst)
    print(f"✓ copied smplx_fitted/ ({len(os.listdir(dst))} files)")
elif os.path.exists(dst):
    print(f"· exists smplx_fitted/ ({len(os.listdir(dst))} files)")
else:
    print(f"✗ smplx_fitted not found in staging")

# a1s1_uv.obj
src = f"{STAGING}/mpmavatar_data/data/a1_s1/a1s1_uv.obj"
dst = f"{REPO_DIR}/data/a1_s1/a1s1_uv.obj"
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy2(src, dst)
    print(f"✓ copied a1s1_uv.obj")
elif os.path.exists(dst):
    print(f"· exists a1s1_uv.obj")
else:
    print(f"✗ a1s1_uv.obj not found in staging")

print("\n✓ All files ready — proceed to Cell 11.")


In [ ]:
import os, glob

STUDIO     = "/teamspace/studios/this_studio"
PRETRAINED = f"{STUDIO}/pretrained_models"
DATA_DIR   = f"{STUDIO}/data"
REPO_DIR   = f"{STUDIO}/mpm_avatar_vds"

PLY_DST      = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
AOMAP_DST    = f"{TRACKING_DST}/aomap"
A1S1_DST     = f"{DATA_DIR}/a1_s1"
SMPLX_DST    = f"{DATA_DIR}/body_models/smplx"
BODY_DST     = f"{DATA_DIR}/body_models"

required = {
    "point_cloud.ply":          f"{PLY_DST}/point_cloud.ply",
    "verts_offset.npy":         f"{PLY_DST}/verts_offset.npy",
    "cams.npz":                 f"{PLY_DST}/cams.npz",
    "shadow_net.pt":            f"{PLY_DST}/shadow_net.pt",
    "params_460.npz":           f"{TRACKING_DST}/params_460.npz",
    "aomap/mesh_cloth_460.png": f"{AOMAP_DST}/mesh_cloth_460.png",
    "split_idx.npz":            f"{A1S1_DST}/split_idx.npz",
    "cam_info.json":            f"{A1S1_DST}/cam_info.json",
    "optimized_weights.npy":    f"{A1S1_DST}/optimized_weights.npy",
    "smplx_fitted/ (dir)":      f"{A1S1_DST}/smplx_fitted",
    "SMPLX_NEUTRAL.npz":        f"{SMPLX_DST}/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":             f"{BODY_DST}/TR00_E096.pt",
    "a1s1_uv.obj":              f"{REPO_DIR}/data/a1_s1/a1s1_uv.obj",
}

all_ok = True
for name, path in required.items():
    ok = os.path.exists(path)
    if not ok:
        all_ok = False
    print(f"  {'✓' if ok else '✗'} {name}")

print(f"\n  params_*.npz : {len(glob.glob(TRACKING_DST+'/params_*.npz'))}")
print(f"  aomap/*.png  : {len(glob.glob(AOMAP_DST+'/*.png'))}")
print(f"  smplx_fitted : {len(os.listdir(A1S1_DST+'/smplx_fitted')) if os.path.exists(A1S1_DST+'/smplx_fitted') else 0} files")
print()
print("✓ All files present — ready for Cell 11!" if all_ok else "✗ Some files missing — check above.")


## Cell 11 — Config
Set all paths and training parameters here.

In [ ]:
import os

STUDIO     = "/teamspace/studios/this_studio"
REPO_DIR   = f"{STUDIO}/mpm_avatar_vds"
PRETRAINED = f"{STUDIO}/pretrained_models"

# ── Paths ───────────────────────────────────────────────────────────────────────────────
TRAINED_MODEL_PATH = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
MODEL_PATH         = f"{PRETRAINED}/model/a1_s1"
DATASET_DIR        = f"{STUDIO}/data"
SPLIT_IDX_PATH     = f"{STUDIO}/data/a1_s1/split_idx.npz"

OUTPUT_DIR   = f"{STUDIO}/output"
WAN_CKPT_DIR = f"{STUDIO}/wan_5b_model"
SDS_CFG      = f"{REPO_DIR}/bridge_sds/configs/sds_test.yaml"

# ── Dataset ───────────────────────────────────────────────────────────────────────────
ACTOR             = 1
SEQUENCE          = 1
TRAIN_FRAME_START = 460
TRAIN_FRAME_NUM   = 16
VERTS_START_IDX   = 460

# ── Training ───────────────────────────────────────────────────────────────────────────
ITERATIONS    = 1000
WANDB_PROJECT = "mpmavatarEval"
WANDB_ENTITY  = "js-teamm"
USE_WANDB     = True
WANDB_API_KEY = ""        # paste your API key here

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO_DIR)

if WANDB_API_KEY:
    import wandb; wandb.login(key=WANDB_API_KEY, relogin=True)
    print("W&B logged in")

print(f"WAN_CKPT_DIR : {WAN_CKPT_DIR}")
print(f"W&B entity   : {WANDB_ENTITY}  project: {WANDB_PROJECT}")
print("Config ready.")


## Cell 12 — Verify All Files Before Training

In [ ]:
import os

checks = {
    "train_sds_physics.py":    f"{REPO_DIR}/train_sds_physics.py",
    "bridge_sds/wan22":        f"{REPO_DIR}/bridge_sds/wan22_i2v_guidance.py",
    "SDS config":              SDS_CFG,
    "point_cloud.ply":         f"{MODEL_PATH}/point_cloud/timestep_030000/point_cloud.ply",
    "verts_offset.npy":        f"{MODEL_PATH}/point_cloud/timestep_030000/verts_offset.npy",
    "cams.npz":                f"{MODEL_PATH}/point_cloud/timestep_030000/cams.npz",
    "shadow_net.pt":           f"{MODEL_PATH}/point_cloud/timestep_030000/shadow_net.pt",
    "params_460.npz":          f"{TRAINED_MODEL_PATH}/params_460.npz",
    "aomap/mesh_cloth_460.png":f"{TRAINED_MODEL_PATH}/aomap/mesh_cloth_460.png",
    "split_idx.npz":           SPLIT_IDX_PATH,
    "cam_info.json":           f"{DATASET_DIR}/a1_s1/cam_info.json",
    "optimized_weights.npy":   f"{DATASET_DIR}/a1_s1/optimized_weights.npy",
    "SMPLX_NEUTRAL.npz":       f"{DATASET_DIR}/body_models/smplx/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":            f"{DATASET_DIR}/body_models/TR00_E096.pt",
    # Wan 5B diffusers layout
    "Wan model_index.json":    f"{WAN_CKPT_DIR}/model_index.json",
    "Wan transformer/":        f"{WAN_CKPT_DIR}/transformer",
    "Wan vae/":                f"{WAN_CKPT_DIR}/vae",
    "Wan text_encoder/":       f"{WAN_CKPT_DIR}/text_encoder",
    "Wan tokenizer/":          f"{WAN_CKPT_DIR}/tokenizer",
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'\u2713' if ok else '\u2717'} {name}")
print()
print("\u2713 All checks passed \u2014 ready to train!" if all_ok else "\u2717 Fix missing files before running Cell 13")


## Cell 13 — Run SDS Physics Training

In [ ]:
## Cell 13a — Start FRESH (fully random init, ignores any checkpoint)
import os, sys
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

wandb_flag     = f"--use_wandb --wandb_entity {WANDB_ENTITY}" if USE_WANDB else ""
split_idx_flag = f"--split_idx_path {SPLIT_IDX_PATH}" if os.path.exists(SPLIT_IDX_PATH) else ""

cmd = (
    f"python train_sds_physics.py "
    f"--trained_model_path {TRAINED_MODEL_PATH} "
    f"--model_path         {MODEL_PATH} "
    f"--dataset_dir        {DATASET_DIR} "
    f"{split_idx_flag} "
    f"--output_dir         {OUTPUT_DIR} "
    f"--actor              {ACTOR} "
    f"--sequence           {SEQUENCE} "
    f"--train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} "
    f"--verts_start_idx    {VERTS_START_IDX} "
    f"--wan_ckpt_dir       {WAN_CKPT_DIR} "
    f"--sds_cfg            {SDS_CFG} "
    f"--iterations         {ITERATIONS} "
    f"--save_name          sds_test "
    f"--random_init_params "
    f"{wandb_flag} "
    f"--wandb_project      {WANDB_PROJECT}"
)
print("FRESH RUN (random init)\n" + cmd.replace(" --", " \\\n    --"))
print("\n" + "="*60 + "\n")
os.system(cmd)


In [ ]:
## Cell 13b — RESUME from latest checkpoint (exact step, params, optimizer)
import os, sys, torch
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

wandb_flag     = f"--use_wandb --wandb_entity {WANDB_ENTITY}" if USE_WANDB else ""
split_idx_flag = f"--split_idx_path {SPLIT_IDX_PATH}" if os.path.exists(SPLIT_IDX_PATH) else ""

resume_pt = f"{OUTPUT_DIR}/sds_test/seed0/resume_latest.pt"
if not os.path.exists(resume_pt):
    print(f"✗ No checkpoint found at {resume_pt}")
    print("  Run Cell 13a first to generate a checkpoint (saved every 100 steps).")
else:
    ckpt = torch.load(resume_pt, map_location="cpu")
    print(f"✓ Resuming from step {ckpt['step']}  "
          f"D={ckpt['D']:.4f}  E={ckpt['E']*100:.1f}Pa  "
          f"H={ckpt['H']:.4f}  friction={ckpt['friction']:.4f}")

    cmd = (
        f"python train_sds_physics.py "
        f"--trained_model_path {TRAINED_MODEL_PATH} "
        f"--model_path         {MODEL_PATH} "
        f"--dataset_dir        {DATASET_DIR} "
        f"{split_idx_flag} "
        f"--output_dir         {OUTPUT_DIR} "
        f"--actor              {ACTOR} "
        f"--sequence           {SEQUENCE} "
        f"--train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} "
        f"--verts_start_idx    {VERTS_START_IDX} "
        f"--wan_ckpt_dir       {WAN_CKPT_DIR} "
        f"--sds_cfg            {SDS_CFG} "
        f"--iterations         {ITERATIONS} "
        f"--save_name          sds_test "
        f"--resume_ckpt        {resume_pt} "
        f"{wandb_flag} "
        f"--wandb_project      {WANDB_PROJECT}"
    )
    print("RESUME RUN\n" + cmd.replace(" --", " \\\n    --"))
    print("\n" + "="*60 + "\n")
    os.system(cmd)


## Cell 14 — Monitor Training Output

In [ ]:
import os, glob

output_dir = f"{OUTPUT_DIR}/sds_test"
print(f"Output dir: {output_dir}")

csv_files = glob.glob(f"{output_dir}/**/*.csv", recursive=True)
npz_files = glob.glob(f"{output_dir}/**/*.npz", recursive=True)
gif_files = glob.glob(f"{output_dir}/**/*.gif", recursive=True)

print(f"CSV files  : {len(csv_files)}")
print(f"NPZ files  : {len(npz_files)}")
print(f"GIF files  : {len(gif_files)}")

if csv_files:
    import pandas as pd
    df = pd.read_csv(csv_files[-1])
    print(f"\nLatest CSV ({os.path.basename(csv_files[-1])}):")
    print(df.tail(10).to_string())


In [ ]:
"""
Cell 15 — Render a video from the best params found in the last run.
Loads sds_best_param_*.npz, runs the simulation, saves a GIF you can view inline.
"""
import os, glob, numpy as np, torch
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Find latest best-params file ─────────────────────────────────────────────
output_root = f"{OUTPUT_DIR}/sds_test"
npz_files   = sorted(glob.glob(f"{output_root}/**/*.npz", recursive=True))
if not npz_files:
    print("No .npz files found — run training first.")
else:
    latest = npz_files[-1]
    best   = dict(np.load(latest))
    print(f"Loaded: {latest}")
    print(f"  D        = {best['D']:.4f}")
    print(f"  E        = {best['E']:.2f} Pa")
    print(f"  H        = {best['H']:.4f}")
    print(f"  friction = {best.get('friction', 0.3):.4f}")
    print(f"  SDS loss = {best['loss']:.6f}  (step {int(best['step'])})")


In [ ]:
"""
Cell 16 — Actually render the simulation video with those params.
Reuses the same SDSPhysicsTrainer (reloads everything), then renders a GIF.
~2-3 min to load, ~1 min to simulate+render.
"""
import yaml, imageio, numpy as np
from pathlib import Path

SDS_CFG_DICT = yaml.safe_load(open(SDS_CFG))

# ── Parse args the same way Cell 13 does ─────────────────────────────────────
import sys, argparse
from arguments import ModelParams, PipelineParams, OptimizationParams

parser = argparse.ArgumentParser()
lp = ModelParams(parser); op = OptimizationParams(parser); pp = PipelineParams(parser)
# Note: --random_init_params is already registered by OptimizationParams above
parser.add_argument("--wan_ckpt_dir", type=str, default=WAN_CKPT_DIR)
parser.add_argument("--wan_repo_root", type=str, default=None)
parser.add_argument("--sds_cfg", type=str, default=SDS_CFG)
parser.add_argument("--min_friction", type=float, default=None)
parser.add_argument("--max_friction", type=float, default=None)
parser.add_argument("--run_eval", action="store_true", default=False)
parser.add_argument("--skip_sim", action="store_true", default=False)
parser.add_argument("--skip_render", action="store_true", default=False)
parser.add_argument("--skip_video", action="store_true", default=False)
parser.add_argument("--local_rank", type=int, default=-1)

raw_args = (
    f"--trained_model_path {TRAINED_MODEL_PATH} "
    f"--model_path {MODEL_PATH} "
    f"--dataset_dir {DATASET_DIR} "
    f"--split_idx_path {SPLIT_IDX_PATH} "
    f"--output_dir {OUTPUT_DIR} "
    f"--actor {ACTOR} --sequence {SEQUENCE} "
    f"--train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} "
    f"--verts_start_idx {VERTS_START_IDX} "
    f"--save_name sds_render"
).split()

raw = parser.parse_args(raw_args)
args, opt, pipe = lp.extract(raw), op.extract(raw), pp.extract(raw)

# Apply YAML overrides
mpm_cfg = SDS_CFG_DICT.get("mpm", {})
if "substep"   in mpm_cfg: args.substep   = int(mpm_cfg["substep"])
if "grid_size" in mpm_cfg: args.grid_size = int(mpm_cfg["grid_size"])

print("Loading trainer (no Wan model needed for rendering)...")

# Load base Trainer only — skip Wan to save VRAM+time
from train_material_params import Trainer
trainer = Trainer(args, opt, pipe, run_eval=False)
print("Trainer loaded.")

# ── Inject best params ────────────────────────────────────────────────────────
D_best        = float(best["D"])
E_best_stored = float(best["E"]) / 100.0    # stored as Pa, convert back to E_stored
H_best        = float(best["H"])
f_best        = float(best.get("friction", 0.3))

trainer.torch_param["D"].data.fill_(D_best)
trainer.torch_param["E"].data.fill_(E_best_stored)
trainer.torch_param["H"].data.fill_(H_best)

# friction: set on mesh colliders
for collider in trainer.mpm_solver.mesh_collider_params:
    collider.friction = f_best

print(f"Params set: D={D_best:.4f}  E={E_best_stored*100:.1f}Pa  H={H_best:.4f}  friction={f_best:.4f}")


In [ ]:
"""
Cell 17 — Simulate + render multi-camera GIF (all test cameras side by side).
Each column = one camera angle. Takes ~1-2 min.
"""
import torch, numpy as np, imageio
from IPython.display import Image as IPyImage, display
import torch.nn.functional as F
from gaussian_renderer import render
from train_material_params import convert_SH

device = "cuda"
bg_color = [1, 1, 1] if trainer.scene.white_bkgd else [0, 0, 0]
bg = torch.tensor(bg_color, dtype=torch.float32, device=device)

# ── Use ALL test cameras ──────────────────────────────────────────────────────
test_cams = list(zip(trainer.scene.test_dataset.camera_list,
                     trainer.scene.test_camera_index))
print(f"Rendering from {len(test_cams)} cameras: {trainer.scene.test_camera_index}")

ao_approx = trainer.gaussians.ao_maps.mean(dim=0)
trainer.gaussians.shadow_net.eval().requires_grad_(False)

@torch.no_grad()
def render_frame(verts, camera, camera_idx):
    trainer.gaussians.set_mesh_by_verts(verts)
    shadow_map = trainer.gaussians.shadow_net(ao_approx)["shadow_map"]
    shadow = F.grid_sample(
        shadow_map, trainer.gaussians.uv_coord,
        mode="bilinear", align_corners=False
    ).squeeze()[..., None][trainer.gaussians.binding]
    colors_precomp = shadow * convert_SH(
        trainer.gaussians.get_features, camera,
        trainer.gaussians, trainer.gaussians.get_xyz,
    )
    render_pkg = render(camera, trainer.gaussians, trainer.pipe, bg,
                        override_color=colors_precomp)
    rendering = (
        render_pkg["render"]
        * torch.exp(trainer.gaussians.cam_m[camera_idx])[:, None, None]
        + trainer.gaussians.cam_c[camera_idx][:, None, None]
    )
    rendering = rendering * render_pkg["mask"]
    if trainer.scene.white_bkgd:
        rendering = rendering + (1.0 - render_pkg["mask"])
    return rendering.clamp(0, 1).cpu()

# ── Reset + simulate MPM ──────────────────────────────────────────────────────
import warp as wp

D = float(trainer.torch_param["D"].item())
E = float(trainer.torch_param["E"].item())
H = float(trainer.torch_param["H"].item())

particle_R_inv = trainer.compute_rest_dir_inv_from_vf(
    torch.stack([
        trainer.vertices_init_position[:, 0],
        trainer.vertices_init_position[:, 1] * H,
        trainer.vertices_init_position[:, 2],
    ], dim=1),
    trainer.new_cloth_faces,
)
trainer.mpm_state.reset_state(
    trainer.n_vertices,
    trainer.particle_init_position.clone(),
    trainer.particle_init_dir.clone(),
    None,
    trainer.particle_init_velo.clone(),
    tensor_R_inv=particle_R_inv.clone(),
    device=device, requires_grad=False,
)

density = torch.ones_like(trainer.particle_init_position[..., 0]) * D
youngs  = torch.ones_like(trainer.particle_init_position[..., 0]) * E * 100.0
trainer.mpm_state.reset_density(density, None, device, update_mass=True)
trainer.mpm_solver.set_E_nu_from_torch(
    trainer.mpm_model, youngs,
    trainer.poisson_ratio.detach().clone(),
    trainer.gamma.detach().clone(),
    trainer.kappa.detach().clone(),
    device,
)
trainer.mpm_solver.prepare_mu_lam(trainer.mpm_model, trainer.mpm_state, device)

delta_time   = 1.0 / 25.0
substep_size = delta_time / trainer.args.substep
num_substeps = int(delta_time / substep_size)
n_frames     = trainer.scene.train_frame_num - 1

# per_cam_frames[cam_i][frame_i] = np.uint8 [H, W, 3]
per_cam_frames = [[] for _ in test_cams]

print(f"Simulating {n_frames} frames …")
for i in range(n_frames):
    mesh_x = trainer.wld2sim(trainer.train_frame_smplx[i].clone())
    mesh_v = trainer.train_frame_smplx_velo[i].clone() * trainer.scale
    jvv    = trainer.train_frame_verts_velo[i, trainer.joint_v_idx].clone() * trainer.scale
    jfv    = jvv[trainer.new_cloth_faces[:trainer.num_joint_f]].mean(1).clone()

    for sub in range(num_substeps):
        trainer.mpm_solver.p2g2p(
            trainer.mpm_model, trainer.mpm_state, substep_size,
            mesh_x=mesh_x + substep_size * sub * mesh_v,
            mesh_v=mesh_v,
            joint_traditional_v=None,
            joint_verts_v=jvv,
            joint_faces_v=jfv,
            device=device,
        )

    particle_pos = wp.to_torch(trainer.mpm_state.particle_x).clone()
    cloth_verts  = trainer.sim2wld(particle_pos[trainer.n_elements:])
    verts = trainer.train_frame_verts[i + 1].clone()
    verts[trainer.reordered_cloth_v_idx] = cloth_verts

    for ci, (cam, cidx) in enumerate(test_cams):
        frame = render_frame(verts, cam, cidx)
        per_cam_frames[ci].append(
            (frame.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        )

    if (i + 1) % 5 == 0:
        print(f"  frame {i+1}/{n_frames}")

# ── Stitch cameras side-by-side and resize to 512px tall ─────────────────────
TARGET_H = 512
gif_frames = []
for i in range(n_frames):
    strips = []
    for ci in range(len(test_cams)):
        img = per_cam_frames[ci][i]              # [H, W, 3]
        h, w = img.shape[:2]
        new_w = int(w * TARGET_H / h)
        # resize with PIL for clean output
        from PIL import Image as _PIL
        img_resized = np.array(
            _PIL.fromarray(img).resize((new_w, TARGET_H), _PIL.LANCZOS)
        )
        strips.append(img_resized)
        # thin white divider between cameras
        if ci < len(test_cams) - 1:
            strips.append(np.ones((TARGET_H, 4, 3), dtype=np.uint8) * 200)
    gif_frames.append(np.concatenate(strips, axis=1))

# ── Save multi-camera GIF ─────────────────────────────────────────────────────
gif_path = f"{OUTPUT_DIR}/sds_best_params_multicam.gif"
imageio.mimsave(gif_path, gif_frames, fps=10, loop=0)
print(f"\nSaved → {gif_path}  ({len(test_cams)} cameras, {n_frames} frames)")
display(IPyImage(filename=gif_path))
